In [1]:
import os
import time
import platform
from shutil import rmtree
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as image
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

from torchvision import datasets
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

# 1. 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':  # Mac OS
    plt.rc('font', family='AppleGothic')
else:  # Linux (Colab 등)
    plt.rc('font', family='NanumBarunGothic')

# 2. 마이너스 기호(-) 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

load_dotenv()
# 오류 발생시
# sklearn.set_config(display="text")

True

In [2]:
base_path = r"C:\kdt_0900_lsu\ml\resource\datasets"
Cloud1_path = os.path.join(base_path, "Cloud1")

In [13]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

os.makedirs(base_path, exist_ok=True)
api.dataset_download_files(r"yeeeengyu/cloud-classification-dataset", path=base_path, unzip=True)

print("다운로드 완료")

Dataset URL: https://www.kaggle.com/datasets/yeeeengyu/cloud-classification-dataset
다운로드 완료


In [3]:
train_path = os.path.join(base_path, "Cloud1", "train")
val_path = os.path.join(base_path, "Cloud1", "val")
test_path = os.path.join(base_path, "Cloud1", "test")

In [4]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [5]:
# 이미지 데이터 불러오기
train_dataset = ImageFolder(train_path, transform=train_transform)
val_dataset = ImageFolder(val_path, transform=test_transform)
test_dataset = ImageFolder(test_path, transform=test_transform)

In [6]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
elif torch.xpu.is_available():
    device = torch.device("xpu")
else:
    device = torch.device("cpu")

print(f"현재 선택된 장치: {device}")

현재 선택된 장치: cpu


In [7]:
model1 = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

In [8]:
model1

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
  

In [9]:
for param in model1.parameters():
    param.requires_grad = False

In [10]:
num_classes = len(train_dataset.classes)


model1.classifier = nn.Sequential(
    
    nn.Dropout(p=0.2, inplace=False),
    nn.Linear(model1.classifier[1].in_features, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout (0.4), 
    nn.Linear(256, num_classes)


)

In [11]:
def train_one_epoch_multiclass(
    model, dataloader, criterion, optimizer, device = "cpu"
):
    model.train()

    sum_losses = 0.0
    correct = 0
    total = 0

    for i, (X_batch, y_batch) in enumerate(dataloader, 1):

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        y_pred = model(X_batch)

        loss = criterion(y_pred, y_batch)

        loss.backward()
        optimizer.step()

        sum_losses += loss.item() * X_batch.size(0)

        y_pred_index += y_pred.argmax(dim=1)

        correct += (y_pred_index == y_batch).sum().item()
        total += y_batch.size(0)

        if i % 20 ==0:
            batch_acc = (
                (y_pred_index == y_batch).float().mean().item() * 100
            )

            print(
                f"[Batch {i}/{len(dataloader)}"
                f"Loss: {loss.item():.4f},"
                f"Accuracy: {batch_acc:.2f}%"
            )

    avg_loss = sum_losses / total
    avg_acc = correct / total * 100

    return avg_loss, avg_acc

In [12]:
@torch.no_grad()
def evaluate_multiclass(model, dataloader, criterion, device="cpu"):
    model.eval()

    sum_losses = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        y_pred_index = y_pred.argmax(dim=1)
        correct += (y_batch == y_pred_index).sum().item()
        total += y_batch.size(0)

    avg_loss = sum_losses / total
    avg_acc = correct / total * 100

    return avg_loss, avg_acc

In [16]:
def train_model_multiclass(model, moedel_name, train_loader, val_loader, criterion, optimzer,epochs,device, save_base_dir):
    save_dir = os.path.join(save_base_dir, model_name)
    os.makedirs(save_dir, exist_ok=True)

    histroy = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    best_val_acc = 0.0

    print(f"------------{model_name}학습시작-----------------------")

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch_multiclass(model, train_loader, criterion,optimzer, device)

        val_loss, val_acc = evaluate_multiclass(model, val_loader, criterion, device)

        print(
            f"[Epoch {epoch + 1 }/{epoch}]"
            f"Train_Loss: {train_loss:.4f}|"
            f"Train Accuracy: {train_acc:.2f}%"
        )

        print(
            f"[Epoch {epoch + 1}/{etrain_accpochs}]"
            f"Val Loss:{val_loss:.4f}|"
            f"val Accutacy: {val_acc:.2f}%"
        )
        
        history["train_loss"].appnef(train_loss)
        history["val_loss"].appnef(val_loss)
        history["train_acc"].appnef(train_acc)
        history["val_acc"].appnef(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc

            torch.save(
                model.state_dict(),
                os.path.join(save_dir, "best_mode_path")
            )

    torch.save(
        model.state_dict(),
        os.path.join(dave_dir, f"last_mode_{epochs}.pth")
    )


    print(f"----------------{model_name}학습종료------------")
    print(f"최고 검증 정확도: {best_val_acc:.2f}%")
    print(f"모델 저장 위치: {save_dir}")

    return histroy

In [17]:
criterion = nn.CrossEntropyLoss()
optimizer1 = optim.Adam(model1.classifier.parameters(), lr=0.001)

In [22]:
history1 = train_one_epoch_multiclass(
    model=model1,
    model_name = "MobileNetV2-Cloud",
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer1,
    epochs=10,
    device=device,
    save_base_dir = os.path.join(Cloud1_path, "weights")
)

NameError: name 'train_loader' is not defined